# ML-6: Model Packaging & Backend Handoff Artifacts

This notebook packages the ML-4 failure prediction model and ML-5 Isolation Forest anomaly detection model into backend-ready production artifacts.

## Key Objectives:
1. **Package Reusable Pipelines**:
   - `ml/models/failure_pipeline.pkl`
   - `ml/models/anomaly_pipeline.pkl`
2. **Generate Standardized Schemas & Metadata**:
   - `ml/models/feature_schema.json`
   - `ml/models/model_metadata.json`
3. **Enforce Backend API Contract**:
   - Inference accepts snake_case features: `type`, `air_temperature`, `process_temperature`, `rotational_speed`, `torque`, `tool_wear`.
4. **Preserve SHAP Feature Names**:
   - Preserves exact recoverable feature names (11 features) required for SHAP attribution.
5. **Verification & Inference Test**:
   - Reloads `.pkl` artifacts and executes sample inference to verify readiness for backend consumption.

In [8]:
import os
import sys
import json
import joblib
import pickle
import numpy as np
import pandas as pd

# Append src module to sys path
sys.path.append(os.path.abspath("../src"))
from feature_engineering import DomainFeatureEngineer, build_preprocessing_pipeline

## 1. Load Trained Pipelines & Artifacts

We load the trained failure model pipeline (`final_failure_pipeline.joblib`) and anomaly pipeline (`anomaly_pipeline.joblib`) from `ml/models/`.

In [9]:
models_dir = "../models"

failure_joblib_path = os.path.join(models_dir, "final_failure_pipeline.joblib")
anomaly_joblib_path = os.path.join(models_dir, "anomaly_pipeline.joblib")

if not os.path.exists(failure_joblib_path):
    raise FileNotFoundError(f"Failure pipeline not found at {failure_joblib_path}. Run ML-4 first.")
if not os.path.exists(anomaly_joblib_path):
    raise FileNotFoundError(f"Anomaly pipeline not found at {anomaly_joblib_path}. Run ML-5 first.")

failure_bundle = joblib.load(failure_joblib_path)
anomaly_bundle = joblib.load(anomaly_joblib_path)

print("Loaded ML-4 Failure Pipeline Bundle:")
print(f"  • Winning Model: {failure_bundle.get('winning_model_name', 'Trained Model')}")
print(f"  • Decision Threshold: {failure_bundle.get('optimal_threshold', 0.50):.4f}")

print("\nLoaded ML-5 Anomaly Pipeline Bundle:")
print(f"  • Model Type: {anomaly_bundle.get('model_type', 'IsolationForest')}")
print(f"  • Contamination: {anomaly_bundle.get('contamination', 0.035)}")

Loaded ML-4 Failure Pipeline Bundle:
  • Winning Model: Balanced Random Forest
  • Decision Threshold: 0.5000

Loaded ML-5 Anomaly Pipeline Bundle:
  • Model Type: IsolationForest
  • Contamination: 0.035


## 2. Package Backend-Ready Pipelines (.pkl)

We structure complete inference bundles containing `engineer`, `preprocessor`, `model`, `feature_names`, `optimal_threshold`, and input schema, and save them as `failure_pipeline.pkl` and `anomaly_pipeline.pkl`.

In [10]:
api_input_schema = [
    "type",
    "air_temperature",
    "process_temperature",
    "rotational_speed",
    "torque",
    "tool_wear"
]

# ---------------------------------------------------------
# FAILURE PIPELINE
# ---------------------------------------------------------

failure_pipeline = failure_bundle["pipeline"]
winning_model_name = failure_bundle["winning_model_name"]
selected_threshold = failure_bundle.get("selected_threshold", 0.50)
feature_names = failure_bundle["feature_names"]
test_metrics = failure_bundle.get("test_metrics", {})

failure_pipeline_pkl = {
    "pipeline": failure_pipeline,
    "model_name": winning_model_name,
    "selected_threshold": selected_threshold,
    "feature_names": feature_names,
    "input_schema": api_input_schema,
    "test_metrics": test_metrics
}


# ---------------------------------------------------------
# ANOMALY PIPELINE
# ---------------------------------------------------------

# ---------------------------------------------------------
# ANOMALY PIPELINE
# ---------------------------------------------------------

anomaly_model = anomaly_bundle["model"]
anomaly_engineer = anomaly_bundle["engineer"]
anomaly_preprocessor = anomaly_bundle["preprocessor"]
contamination_setting = anomaly_bundle.get("contamination", 0.035)

anomaly_pipeline_pkl = {
    "model": anomaly_model,
    "engineer": anomaly_engineer,
    "preprocessor": anomaly_preprocessor,
    "contamination": contamination_setting,
    "threshold": 0.0,
    "feature_names": feature_names,
    "input_schema": api_input_schema,
    "description": "Isolation Forest Anomaly Pipeline trained on normal operating baseline."
}


# ---------------------------------------------------------
# SAVE PKL FILES
# ---------------------------------------------------------

failure_pkl_path = os.path.join(models_dir, "failure_pipeline.pkl")
anomaly_pkl_path = os.path.join(models_dir, "anomaly_pipeline.pkl")

joblib.dump(failure_pipeline_pkl, failure_pkl_path)
joblib.dump(anomaly_pipeline_pkl, anomaly_pkl_path)

print(f"Exported failure pipeline to: {os.path.abspath(failure_pkl_path)}")
print(f"Exported anomaly pipeline to: {os.path.abspath(anomaly_pkl_path)}")

Exported failure pipeline to: c:\Users\bingu\OneDrive\Desktop\cognizant\predictive-maintenance\ml\models\failure_pipeline.pkl
Exported anomaly pipeline to: c:\Users\bingu\OneDrive\Desktop\cognizant\predictive-maintenance\ml\models\anomaly_pipeline.pkl


## 3. Generate JSON Metadata & Feature Schemas

We generate `feature_schema.json` and `model_metadata.json` detailing the API contract, feature mappings, evaluation metrics, and sample input/output.

In [11]:
# =========================================================
# 1. FEATURE SCHEMA JSON
# =========================================================

feature_schema = {
    "api_input_features": api_input_schema,

    "feature_types": {
        "type": "string (categorical: L, M, H)",
        "air_temperature": "float (Kelvin)",
        "process_temperature": "float (Kelvin)",
        "rotational_speed": "float/int (rpm)",
        "torque": "float (Nm)",
        "tool_wear": "float/int (min)"
    },

    "raw_dataset_columns": [
        "Type",
        "Air temperature [K]",
        "Process temperature [K]",
        "Rotational speed [rpm]",
        "Torque [Nm]",
        "Tool wear [min]"
    ],

    "engineered_domain_features": [
        "temperature_difference",
        "mechanical_power_W",
        "overstrain_index"
    ],

    "preprocessed_model_features": feature_names,

    "target_column": "Machine failure",

    "excluded_columns": [
        "UDI",
        "Product ID",
        "TWF",
        "HDF",
        "PWF",
        "OSF",
        "RNF"
    ]
}

schema_path = os.path.join(models_dir, "feature_schema.json")

with open(schema_path, "w") as f:
    json.dump(feature_schema, f, indent=2)

print(f"Saved feature schema JSON to: {os.path.abspath(schema_path)}")


# =========================================================
# 2. MODEL METADATA JSON
# =========================================================

model_metadata = {
    "failure_model": {
        "model_name": failure_pipeline_pkl["model_name"],
        "artifact_file": "failure_pipeline.pkl",
        "optimal_threshold": selected_threshold,
        "evaluation_metrics": test_metrics
    },

    "anomaly_model": {
        "model_name": "IsolationForest",
        "artifact_file": "anomaly_pipeline.pkl",
        "contamination": contamination_setting,
        "threshold": 0.0,
        "training_strategy": (
            "Trained on normal operating baseline "
            "(Machine failure == 0)"
        )
    },

    "sample_input": {
        "type": "M",
        "air_temperature": 298.5,
        "process_temperature": 308.6,
        "rotational_speed": 1550,
        "torque": 42.3,
        "tool_wear": 120
    },

    "sample_output": {
        "anomaly": {
            "is_anomaly": False,
            "score": 0.1245
        },

        "failure_prediction": {
            "predicted_failure": False,
            "probability": 0.0312,
            "risk_level": "LOW"
        }
    },

    "dependencies": {
        "python": ">=3.10",
        "scikit-learn": ">=1.2.0",
        "joblib": ">=1.2.0",
        "numpy": ">=1.23.0",
        "pandas": ">=1.5.0",
        "xgboost": ">=1.7.0"
    }
}

metadata_path = os.path.join(models_dir, "model_metadata.json")

with open(metadata_path, "w") as f:
    json.dump(model_metadata, f, indent=2)

print(f"Saved model metadata JSON to: {os.path.abspath(metadata_path)}")

Saved feature schema JSON to: c:\Users\bingu\OneDrive\Desktop\cognizant\predictive-maintenance\ml\models\feature_schema.json
Saved model metadata JSON to: c:\Users\bingu\OneDrive\Desktop\cognizant\predictive-maintenance\ml\models\model_metadata.json


## 4. Verification & Sample Inference Test

We reload the packaged `.pkl` artifacts and test inference using the exact API contract input format (`type`, `air_temperature`, `process_temperature`, `rotational_speed`, `torque`, `tool_wear`).

In [12]:
# =========================================================
# LOAD PACKAGED ARTIFACTS
# =========================================================

loaded_failure_bundle = joblib.load(failure_pkl_path)
loaded_anomaly_bundle = joblib.load(anomaly_pkl_path)

# Preprocessor is stored separately
preprocessor_path = os.path.join(models_dir, "preprocessor.joblib")
loaded_preprocessor = joblib.load(preprocessor_path)

print("All artifacts loaded successfully.")

print("\nFailure bundle keys:")
print(loaded_failure_bundle.keys())

print("\nAnomaly bundle keys:")
print(loaded_anomaly_bundle.keys())


# =========================================================
# SAMPLE API INPUT
# =========================================================

sample_records = [
    {
        "type": "M",
        "air_temperature": 298.5,
        "process_temperature": 308.6,
        "rotational_speed": 1550,
        "torque": 42.3,
        "tool_wear": 120
    },
    {
        "type": "L",
        "air_temperature": 302.1,
        "process_temperature": 311.5,
        "rotational_speed": 1250,
        "torque": 68.4,
        "tool_wear": 225
    }
]

input_df = pd.DataFrame(sample_records)

print("\nSample API Input DataFrame:")
print(input_df)


# =========================================================
# PREPARE INPUT USING SAME FEATURE ENGINEERING
# =========================================================

eng_df = input_df.rename(
    columns={
        "type": "Type",
        "air_temperature": "Air temperature [K]",
        "process_temperature": "Process temperature [K]",
        "rotational_speed": "Rotational speed [rpm]",
        "torque": "Torque [Nm]",
        "tool_wear": "Tool wear [min]"
    }
).copy()


# Create the same engineered features used during training
eng_df["temperature_difference"] = (
    eng_df["Process temperature [K]"]
    - eng_df["Air temperature [K]"]
)

eng_df["mechanical_power_W"] = (
    eng_df["Torque [Nm]"]
    * eng_df["Rotational speed [rpm]"]
    * (2 * np.pi / 60.0)
)

eng_df["overstrain_index"] = (
    eng_df["Tool wear [min]"]
    * eng_df["Torque [Nm]"]
)


# =========================================================
# PREPROCESSING
# =========================================================

proc_features = loaded_preprocessor.transform(eng_df)

feature_names = loaded_preprocessor.get_feature_names_out()

proc_df = pd.DataFrame(
    proc_features,
    columns=feature_names
)


# =========================================================
# FAILURE PREDICTION
# =========================================================

failure_pipeline = loaded_failure_bundle["pipeline"]

threshold = loaded_failure_bundle["selected_threshold"]

probs = failure_pipeline.predict_proba(input_df)[:, 1]

preds = (probs >= threshold).astype(bool)


# =========================================================
# ANOMALY DETECTION
# =========================================================

anom_model = loaded_anomaly_bundle["model"]

# Use exact feature names expected by Isolation Forest
proc_anom_df = pd.DataFrame(
    proc_features,
    columns=anom_model.feature_names_in_
)

raw_scores = anom_model.decision_function(proc_anom_df)

anom_preds = (
    anom_model.predict(proc_anom_df) == -1
)


# =========================================================
# VERIFICATION RESULTS
# =========================================================

print("\n--- Verification Inference Results ---")

for i, rec in enumerate(sample_records):

    print(
        f"\nRecord {i+1} "
        f"({rec['type']} | Torque: {rec['torque']} | "
        f"Wear: {rec['tool_wear']}):"
    )

    print(
        f"  Anomaly Status:     "
        f"is_anomaly={anom_preds[i]} "
        f"(score={raw_scores[i]:.4f})"
    )

    print(
        f"  Failure Probability: "
        f"{probs[i]:.4f}"
    )

    print(
        f"  Predicted Failure:   "
        f"{preds[i]} "
        f"(Threshold: {threshold:.4f})"
    )

All artifacts loaded successfully.

Failure bundle keys:
dict_keys(['pipeline', 'model_name', 'selected_threshold', 'feature_names', 'input_schema', 'test_metrics'])

Anomaly bundle keys:
dict_keys(['model', 'engineer', 'preprocessor', 'contamination', 'threshold', 'feature_names', 'input_schema', 'description'])

Sample API Input DataFrame:
  type  air_temperature  process_temperature  rotational_speed  torque  \
0    M            298.5                308.6              1550    42.3   
1    L            302.1                311.5              1250    68.4   

   tool_wear  
0        120  
1        225  

--- Verification Inference Results ---

Record 1 (M | Torque: 42.3 | Wear: 120):
  Anomaly Status:     is_anomaly=False (score=0.0743)
  Failure Probability: 0.0000
  Predicted Failure:   False (Threshold: 0.4900)

Record 2 (L | Torque: 68.4 | Wear: 225):
  Anomaly Status:     is_anomaly=True (score=-0.0670)
  Failure Probability: 0.9900
  Predicted Failure:   True (Threshold: 0.4900)

In [13]:
anom_model = loaded_anomaly_bundle["model"]

print("=== Anomaly Model Feature Names ===")
print(anom_model.feature_names_in_)

print("\n=== Current proc_df Feature Names ===")
print(proc_df.columns.tolist())

print("\n=== Number of Features ===")
print("Anomaly model:", len(anom_model.feature_names_in_))
print("Current:", len(proc_df.columns))

=== Anomaly Model Feature Names ===
['Type_H' 'Type_L' 'Type_M' 'Air temperature [K]'
 'Process temperature [K]' 'Rotational speed [rpm]' 'Torque [Nm]'
 'Tool wear [min]' 'temperature_difference' 'mechanical_power_W'
 'overstrain_index']

=== Current proc_df Feature Names ===
['Type_H', 'Type_L', 'Type_M', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'temperature_difference', 'mechanical_power_W', 'overstrain_index']

=== Number of Features ===
Anomaly model: 11
Current: 11
